# Advanced Document Question Answering System using Retrieval-Augmented Generation (RAG)

## Objective

This project implements an end-to-end Retrieval-Augmented Generation (RAG) system capable of answering user questions from custom documents such as PDFs, TXT files, and DOCX files.

### Workflow

1. Document Ingestion
2. Text Chunking
3. Embedding Generation
4. Vector Database (FAISS)
5. Similarity Search
6. Retrieval
7. Prompt Augmentation
8. Answer Generation using Groq Llama-3
9. Validation
10. Experiments (Chunk Size, Hybrid Search, Re-ranking)


In [1]:
!pip install -q langchain==0.3.13
!pip install -q langchain-community==0.3.13
!pip install -q langchain-core==0.3.28
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q docx2txt
!pip install -q python-dotenv
!pip install -q rank-bm25
!pip install -q pandas
!pip install -q langchain-huggingface==0.1.2
!pip install -q langchain-groq==0.2.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 0.3.28 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.13 requires langsmith<0.3,>=0.1.17, but you have langsmith 0.10.2 which is incompatible.
langchain-community 0.3.13 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.10.2 which is incompatible.


## Import Required Libraries

In [2]:
import os
import time
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain.prompts import PromptTemplate


## Load Groq API Key

In [3]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(GROQ_API_KEY[:15] + "********")


gsk_25QN9p5rwCd********


## Initialize Groq LLM

In [4]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)


In [5]:
response = llm.invoke(
    "Explain Retrieval Augmented Generation."
)

print(response.content)


Retrieval Augmented Generation (RAG) is a type of artificial intelligence (AI) model that combines the strengths of two different approaches: retrieval-based models and generative models.

**Retrieval-based models:**
These models are designed to retrieve relevant information from a large database or knowledge graph. They use techniques such as information retrieval, natural language processing (NLP), and machine learning to find the most relevant information that matches a given query or prompt.

**Generative models:**
These models are designed to generate new text, images, or other forms of content based on a given prompt or input. They use techniques such as language models, neural networks, and machine learning to generate new content that is coherent and relevant.

**Retrieval Augmented Generation (RAG):**
RAG models combine the strengths of both retrieval-based and generative models. They first retrieve relevant information from a large database or knowledge graph using a retrieva

In [6]:
pdf_loader = PyPDFLoader("Resume.pdf")

pdf_docs = pdf_loader.load()

print("PDF Pages:", len(pdf_docs))

PDF Pages: 1


In [7]:
text_docs = []

if os.path.exists("sample_resume.txt"):

    text_loader = TextLoader("sample_resume.txt")

    text_docs = text_loader.load()

    print("TXT Loaded:", len(text_docs))

else:

    print("No TXT Found")

No TXT Found


In [8]:
documents=[]

documents.extend(pdf_docs)

documents.extend(text_docs)

print("Total Documents:",len(documents))

Total Documents: 1


In [9]:
documents[0].page_content[:1000]


'Dhanush Chiraboina\n♂¶ap-¶arker-altHyderabad, India\n♂phone+91-9391544253/envel⌢pedhanushyadav0099@gmail.com/linkedinlinkedin.com/in/dhanush-chiraboina/githubgithub.com/Dhanush04925\nAbout Me\nComputer Science student at CVR College of Engineering with strong foundations in Machine Learning and Deep\nLearning, experienced in building and deploying data-driven applications using Python, with solid DSA and\nproblem-solving skills.\nEducation\nCVR College of EngineeringHyderabad, India\nB.Tech in Computer Science and Engineering (AI & ML) — CGPA: 8.6 2023–2027\nNarayana Junior College\nIntermediate (MPC) — Percentage: 96.9% 2021–2023\nTechnical Skills\nProgramming:Python, Java, SQL\nWeb Technologies:HTML, CSS, JavaScript\nMachine Learning:Supervised Learning, Unsupervised Learning, Feature Engineering, Model Training\nDeep Learning:ANN, CNN, RNN, LSTM, GRU\nGenerative AI:RAG, LangChain (Basic)\nData Science:Data Analysis, Data Visualization\nLibraries:Scikit-learn, TensorFlow, Pandas, Nu

## Document Statistics

In [10]:
lengths = [len(doc.page_content) for doc in documents]

stats = pd.DataFrame({
    "Metric": [
        "Total Documents",
        "Average Length",
        "Maximum Length",
        "Minimum Length"
    ],
    "Value": [
        len(documents),
        sum(lengths) / len(lengths),
        max(lengths),
        min(lengths)
    ]
})

stats


,Metric,Value
0,Total Documents,1.0
1,Average Length,1983.0
2,Maximum Length,1983.0
3,Minimum Length,1983.0


## Text Chunking

The documents are split into overlapping chunks to improve retrieval accuracy.


In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Chunks:", len(chunks))


Chunks: 5


In [12]:
chunks[0].page_content


'Dhanush Chiraboina\n♂¶ap-¶arker-altHyderabad, India\n♂phone+91-9391544253/envel⌢pedhanushyadav0099@gmail.com/linkedinlinkedin.com/in/dhanush-chiraboina/githubgithub.com/Dhanush04925\nAbout Me\nComputer Science student at CVR College of Engineering with strong foundations in Machine Learning and Deep\nLearning, experienced in building and deploying data-driven applications using Python, with solid DSA and\nproblem-solving skills.\nEducation\nCVR College of EngineeringHyderabad, India'

In [13]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

chunk_df = pd.DataFrame({
    "Metric": [
        "Chunks",
        "Average Length",
        "Maximum Length",
        "Minimum Length"
    ],
    "Value": [
        len(chunks),
        round(sum(chunk_lengths) / len(chunk_lengths), 2),
        max(chunk_lengths),
        min(chunk_lengths)
    ]
})

chunk_df


,Metric,Value
0,Chunks,5.0
1,Average Length,440.8
2,Maximum Length,483.0
3,Minimum Length,377.0


## Embedding Model

We use

`sentence-transformers/all-MiniLM-L6-v2`

Embedding Dimension = 384


In [14]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12416.67it/s]


In [15]:
embedding = embedding_model.embed_query("Artificial Intelligence")

print("Embedding Dimension:", len(embedding))


Embedding Dimension: 384


In [16]:
vector_store = FAISS.from_documents(
    chunks,
    embedding_model
)


## Vector Database

`VectorStore` stores the chunk embeddings and configures them for fast cosine-similarity search using FAISS's `IndexFlatIP` over normalized vectors.


In [17]:
vector_store.save_local("vector_store")


In [18]:
vector_store = FAISS.load_local(
    "vector_store",
    embedding_model,
    allow_dangerous_deserialization=True
)


# Query Processing

The user question is converted into an embedding and matched against the vector database to retrieve the most relevant document chunks.


In [19]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever Created Successfully")


Retriever Created Successfully


In [20]:
query = "What is the main objective of the document?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} document chunks")


Retrieved 3 document chunks


In [21]:
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Retrieved Chunk {i}")
    print("=" * 80)
    print(doc.page_content)
    print("\n")


Retrieved Chunk 1
context-aware responses.
• Improved answer accuracy by combining semantic retrieval with LLM generation, reducing hallucinations in responses.
Certifications
Salesforce AgentBlazer– Completed Beginner and Innovator levels
Database Programming with SQL– Oracle Academy
Competitive Programming
Solved150+ problemson LeetCode
Strong understanding of Data Structures and Algorithms


Retrieved Chunk 2
problem-solving skills.
Education
CVR College of EngineeringHyderabad, India
B.Tech in Computer Science and Engineering (AI & ML) — CGPA: 8.6 2023–2027
Narayana Junior College
Intermediate (MPC) — Percentage: 96.9% 2021–2023
Technical Skills
Programming:Python, Java, SQL
Web Technologies:HTML, CSS, JavaScript
Machine Learning:Supervised Learning, Unsupervised Learning, Feature Engineering, Model Training
Deep Learning:ANN, CNN, RNN, LSTM, GRU
Generative AI:RAG, LangChain (Basic)


Retrieved Chunk 3
Dhanush Chiraboina
♂¶ap-¶arker-altHyderabad, India
♂phone+91-9391544253/envel⌢pe

# Prompt Augmentation

The retrieved document chunks are combined with the user's question to create a grounded prompt for the language model.


In [22]:
prompt_template = """
You are a helpful AI assistant.

Answer ONLY using the context below.

If the answer is not available in the context,
respond with:

"I could not find the answer in the uploaded documents."

Context:
{context}

Question:
{question}

Answer:
"""


In [23]:
def rag_qa(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "context": docs
    }


In [24]:
result = rag_qa(
    "What is the objective of this document?"
)

print(result["answer"])


I could not find the answer in the uploaded documents.


In [25]:
result = rag_qa(
    "Explain Retrieval Augmented Generation."
)

print(result["answer"])


Retrieval Augmented Generation (RAG) is a technique used in natural language processing (NLP) that combines the strengths of retrieval-based and generation-based models. 

In RAG, a retrieval model is used to retrieve relevant information from a large corpus of text, and then a generation model is used to generate a response based on the retrieved information. This approach allows the model to leverage the strengths of both retrieval and generation, resulting in more accurate and informative responses.

In the context of the provided information, the user has experience with RAG and LangChain, having engineered a full RAG pipeline using LangChain to process video transcripts, generate embeddings, and retrieve context-aware responses.


In [26]:
for i, doc in enumerate(result["context"], start=1):
    print("=" * 80)
    print(f"Source Chunk {i}")
    print("=" * 80)
    print(doc.page_content)
    print()


Source Chunk 1
context-aware responses.
• Improved answer accuracy by combining semantic retrieval with LLM generation, reducing hallucinations in responses.
Certifications
Salesforce AgentBlazer– Completed Beginner and Innovator levels
Database Programming with SQL– Oracle Academy
Competitive Programming
Solved150+ problemson LeetCode
Strong understanding of Data Structures and Algorithms

Source Chunk 2
historical sales, weather, and exam schedules.
• Deployed an interactiveStreamlitdashboard on cloud withMongoDBbackend, enabling canteen owners to visualize trends
and reduce food wastage through data-driven decisions.
YouTube Chatbot using RAG and LangChain
• Engineered a full RAG pipeline using LangChain — processed video transcripts, generated embeddings, and retrieved
context-aware responses.

Source Chunk 3
problem-solving skills.
Education
CVR College of EngineeringHyderabad, India
B.Tech in Computer Science and Engineering (AI & ML) — CGPA: 8.6 2023–2027
Narayana Junior College

# Validation

Testing the RAG pipeline using multiple dynamic questions.


In [27]:
questions = [
    "What is the candidate's name?",
    "What is the highest educational qualification?",
    "Which programming languages does the candidate know?",
    "What machine learning projects has the candidate completed?",
    "What technical skills are listed in the resume?",
    "What internships has the candidate completed?",
    "Which certifications are mentioned?",
    "What are the candidate's contact details?",
    "What tools and frameworks has the candidate worked with?",
    "Summarize the candidate's profile."
]


## Validation Logs — Dynamic Sample Questions

We run a small suite of test questions, log the retrieved chunk count and the generated answer, and print an end-to-end validation report.


In [28]:
validation_logs = []

for q in questions:

    start = time.time()

    response = rag_qa(q)

    end = time.time()

    validation_logs.append({
        "Question": q,
        "Answer": response["answer"],
        "Retrieved Chunks": len(response["context"]),
        "Time (sec)": round(end - start, 2)
    })

validation_df = pd.DataFrame(validation_logs)

validation_df


,Question,Answer,Retrieved Chunks,Time (sec)
0,What is the candidate's name?,Dhanush Chiraboina,3,0.15
1,What is the highest educational qualification?,B.Tech in Computer Science and Engineering (AI...,3,0.31
2,Which programming languages does the candidate...,The candidate knows the following programming ...,3,0.15
3,What machine learning projects has the candida...,The candidate has completed the following mach...,3,0.18
4,What technical skills are listed in the resume?,The technical skills listed in the resume are:...,3,0.27
5,What internships has the candidate completed?,I could not find the answer in the uploaded do...,3,0.16
6,Which certifications are mentioned?,Salesforce AgentBlazer and Database Programmin...,3,0.55
7,What are the candidate's contact details?,The candidate's contact details are:\n\n- Phon...,3,0.18
8,What tools and frameworks has the candidate wo...,The candidate has worked with the following to...,3,0.45
9,Summarize the candidate's profile.,"The candidate, Dhanush Chiraboina, is a Comput...",3,0.21


In [29]:
validation_df.to_csv(
    "validation_logs.csv",
    index=False
)

validation_df.head()


,Question,Answer,Retrieved Chunks,Time (sec)
0,What is the candidate's name?,Dhanush Chiraboina,3,0.15
1,What is the highest educational qualification?,B.Tech in Computer Science and Engineering (AI...,3,0.31
2,Which programming languages does the candidate...,The candidate knows the following programming ...,3,0.15
3,What machine learning projects has the candida...,The candidate has completed the following mach...,3,0.18
4,What technical skills are listed in the resume?,The technical skills listed in the resume are:...,3,0.27


# System Metrics Report

In [30]:
metrics = pd.DataFrame({
    "Metric": [
        "Chunk Size",
        "Chunk Overlap",
        "Embedding Model",
        "Embedding Dimension",
        "Vector Store",
        "Retriever",
        "Language Model"
    ],
    "Value": [
        500,
        100,
        "all-MiniLM-L6-v2",
        384,
        "FAISS",
        "Similarity Search (Top-3)",
        "Groq Llama-3.1-8B-Instant"
    ]
})

metrics


,Metric,Value
0,Chunk Size,500
1,Chunk Overlap,100
2,Embedding Model,all-MiniLM-L6-v2
3,Embedding Dimension,384
4,Vector Store,FAISS
5,Retriever,Similarity Search (Top-3)
6,Language Model,Groq Llama-3.1-8B-Instant


In [31]:
metrics.to_csv(
    "system_metrics.csv",
    index=False
)

metrics


,Metric,Value
0,Chunk Size,500
1,Chunk Overlap,100
2,Embedding Model,all-MiniLM-L6-v2
3,Embedding Dimension,384
4,Vector Store,FAISS
5,Retriever,Similarity Search (Top-3)
6,Language Model,Groq Llama-3.1-8B-Instant


# Experiment 1: Chunk Size Comparison

The objective is to evaluate how different chunk sizes affect retrieval quality and response generation.


In [32]:
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

small_chunks = small_splitter.split_documents(documents)

print("Chunks:", len(small_chunks))


Chunks: 8


In [33]:
small_vector = FAISS.from_documents(
    small_chunks,
    embedding_model
)

small_retriever = small_vector.as_retriever(
    search_kwargs={"k": 3}
)


In [34]:
query = "Summarize the candidate's skills"

docs = small_retriever.invoke(query)

for d in docs:
    print(d.page_content[:400])
    print()


Narayana Junior College
Intermediate (MPC) — Percentage: 96.9% 2021–2023
Technical Skills
Programming:Python, Java, SQL
Web Technologies:HTML, CSS, JavaScript
Machine Learning:Supervised Learning, Unsupervised Learning, Feature Engineering, Model Training
Deep Learning:ANN, CNN, RNN, LSTM, GRU

Certifications
Salesforce AgentBlazer– Completed Beginner and Innovator levels
Database Programming with SQL– Oracle Academy
Competitive Programming
Solved150+ problemson LeetCode
Strong understanding of Data Structures and Algorithms

Dhanush Chiraboina
♂¶ap-¶arker-altHyderabad, India
♂phone+91-9391544253/envel⌢pedhanushyadav0099@gmail.com/linkedinlinkedin.com/in/dhanush-chiraboina/githubgithub.com/Dhanush04925
About Me
Computer Science student at CVR College of Engineering with strong foundations in Machine Learning and Deep



### Observation

Smaller chunks improve precision but increase the total number of vectors stored.


In [35]:
large_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

large_chunks = large_splitter.split_documents(documents)

print(len(large_chunks))


4


In [36]:
large_vector = FAISS.from_documents(
    large_chunks,
    embedding_model
)

large_retriever = large_vector.as_retriever(
    search_kwargs={"k": 3}
)


In [37]:
docs = large_retriever.invoke(query)

for d in docs:
    print(d.page_content[:400])
    print()


Dhanush Chiraboina
♂¶ap-¶arker-altHyderabad, India
♂phone+91-9391544253/envel⌢pedhanushyadav0099@gmail.com/linkedinlinkedin.com/in/dhanush-chiraboina/githubgithub.com/Dhanush04925
About Me
Computer Science student at CVR College of Engineering with strong foundations in Machine Learning and Deep
Learning, experienced in building and deploying data-driven applications using Python, with solid DSA a

historical sales, weather, and exam schedules.
• Deployed an interactiveStreamlitdashboard on cloud withMongoDBbackend, enabling canteen owners to visualize trends
and reduce food wastage through data-driven decisions.
YouTube Chatbot using RAG and LangChain
• Engineered a full RAG pipeline using LangChain — processed video transcripts, generated embeddings, and retrieved
context-aware responses.


Intermediate (MPC) — Percentage: 96.9% 2021–2023
Technical Skills
Programming:Python, Java, SQL
Web Technologies:HTML, CSS, JavaScript
Machine Learning:Supervised Learning, Unsupervised Learning, 

### Observation

Larger chunks preserve context but may retrieve irrelevant information.


In [38]:
comparison = pd.DataFrame({
    "Chunk Size": [300, 500, 700],
    "Overlap": [50, 100, 100],
    "Advantages": [
        "Higher Precision",
        "Balanced",
        "Better Context"
    ],
    "Disadvantages": [
        "More Chunks",
        "Balanced",
        "Lower Precision"
    ]
})

comparison


,Chunk Size,Overlap,Advantages,Disadvantages
0,300,50,Higher Precision,More Chunks
1,500,100,Balanced,Balanced
2,700,100,Better Context,Lower Precision


# Experiment 2: Hybrid Search

Combine

1. Keyword Search (BM25)
2. Vector Search (FAISS)


In [39]:
from rank_bm25 import BM25Okapi


In [40]:
tokenized_chunks = [
    doc.page_content.split()
    for doc in chunks
]

bm25 = BM25Okapi(tokenized_chunks)


In [41]:
query = "Python skills"

scores = bm25.get_scores(query.split())

best_index = scores.argmax()

print(chunks[best_index].page_content)


Dhanush Chiraboina
♂¶ap-¶arker-altHyderabad, India
♂phone+91-9391544253/envel⌢pedhanushyadav0099@gmail.com/linkedinlinkedin.com/in/dhanush-chiraboina/githubgithub.com/Dhanush04925
About Me
Computer Science student at CVR College of Engineering with strong foundations in Machine Learning and Deep
Learning, experienced in building and deploying data-driven applications using Python, with solid DSA and
problem-solving skills.
Education
CVR College of EngineeringHyderabad, India


Observation

BM25 performs lexical matching.

FAISS performs semantic matching.

Combining both improves retrieval quality.


### Re-ranking

In [42]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-TinyBERT-L-2-v2"
)


Loading weights: 100%|██████████| 41/41 [00:00<00:00, 15179.32it/s]


In [43]:
query = "Explain candidate projects"

docs = retriever.invoke(query)


In [44]:
pairs = [
    (query, d.page_content)
    for d in docs
]

scores = reranker.predict(pairs)


In [45]:
ranked = sorted(
    zip(scores, docs),
    reverse=True,
    key=lambda x: x[0]
)


In [46]:
for score, doc in ranked:
    print("Score:", score)
    print(doc.page_content[:400])
    print()


Score: -10.941462
Deep Learning:ANN, CNN, RNN, LSTM, GRU
Generative AI:RAG, LangChain (Basic)
Data Science:Data Analysis, Data Visualization
Libraries:Scikit-learn, TensorFlow, Pandas, NumPy, Matplotlib, NLTK, OpenCV
Tools:Git, GitHub, Jupyter Notebook, Google Colab
Projects
Smart Canteen Management Decision Support System
• Developed an ML-based demand forecasting system usingRandom Forestto predict next-day food 

Score: -11.216891
problem-solving skills.
Education
CVR College of EngineeringHyderabad, India
B.Tech in Computer Science and Engineering (AI & ML) — CGPA: 8.6 2023–2027
Narayana Junior College
Intermediate (MPC) — Percentage: 96.9% 2021–2023
Technical Skills
Programming:Python, Java, SQL
Web Technologies:HTML, CSS, JavaScript
Machine Learning:Supervised Learning, Unsupervised Learning, Feature Engineering, Model T

Score: -11.35604
Dhanush Chiraboina
♂¶ap-¶arker-altHyderabad, India
♂phone+91-9391544253/envel⌢pedhanushyadav0099@gmail.com/linkedinlinkedin.com/in/dhanush-chir

Observation

The CrossEncoder re-ranking model improves retrieval quality by ordering retrieved chunks according to semantic relevance before passing them to the language model.


In [47]:
report = pd.DataFrame({
    "Metric": [
        "Documents",
        "Chunks",
        "Chunk Size",
        "Chunk Overlap",
        "Embedding Model",
        "Embedding Dimension",
        "Vector Store",
        "Retriever",
        "LLM"
    ],
    "Value": [
        len(documents),
        len(chunks),
        500,
        100,
        "all-MiniLM-L6-v2",
        384,
        "FAISS",
        "Similarity Search",
        "Groq Llama-3.1-8B"
    ]
})

report


,Metric,Value
0,Documents,1
1,Chunks,5
2,Chunk Size,500
3,Chunk Overlap,100
4,Embedding Model,all-MiniLM-L6-v2
5,Embedding Dimension,384
6,Vector Store,FAISS
7,Retriever,Similarity Search
8,LLM,Groq Llama-3.1-8B


In [48]:
report.to_csv(
    "system_metrics.csv",
    index=False
)


# Conclusion

The developed Retrieval-Augmented Generation (RAG) system successfully performs document question answering over custom documents.

Achievements:

- Generated semantic embeddings using a pre-trained sentence transformer.
- Stored embeddings in a FAISS vector database.
- Retrieved relevant document chunks using similarity search.
- Generated grounded answers using the Groq Llama-3.1 model.
- Evaluated retrieval performance using validation questions.
- Compared different chunk sizes.
- Implemented hybrid retrieval using BM25 and FAISS.
- Improved retrieval relevance using CrossEncoder re-ranking.

